# Limpeza e Tratamento de Dados Educacionais do Brasil

Este notebook realiza a limpeza e tratamento completo dos dados educacionais do Brasil

## Estrutura do Processo
1. Importação e configuração das bibliotecas
2. Limpeza individual de cada conjunto de dados
3. Tratamento de valores ausentes, incorretos e padronização
4. Exportação dos dados processados

In [1]:
import pandas as pd
import pyarrow as arrow
import ydata as yd
import numpy as np
import openpyxl

### Conversão das tabelas XLS para CSV

In [ ]:
# Definição do caminho dos arquivos
deslocamento_path = '/home/teodoro/Documents/ZettaLab/ZettaLab-Dados/data/Raw/Deslocamento.xlsx'
moradia_path = '/home/teodoro/Documents/ZettaLab/ZettaLab-Dados/data/Raw/Moradia.xlsx'

# Leitura dos arquivos XLS usando pandas
deslocamento_df = pd.read_excel(deslocamento_path)
moradia_df = pd.read_excel(moradia_path)

# Salvando os DataFrames como arquivos CSV
deslocamento_df.to_csv('/home/teodoro/Documents/ZettaLab/ZettaLab-Dados/data/Raw/Deslocamento.csv', index=False)
moradia_df.to_csv('/home/teodoro/Documents/ZettaLab/ZettaLab-Dados/data/Raw/Moradia.csv', index=False)

In [ ]:
### Removendo os arquivos XLS originais
# !rm /home/teodoro/Documents/ZettaLab/ZettaLab-Dados/data/Raw/Deslocamento.xlsx
# !rm /home/teodoro/Documents/ZettaLab/ZettaLab-Dados/data/Raw/Moradia.xlsx

### Este trecho foi dedicado ao processamento do arquivo Deslocamento.csv

In [11]:
import pandas as pd
import numpy as np
import io

input_file_path = '../data/Raw/Deslocamento.csv'
output_filename = '../data/Processed/Deslocamento_processado_10_a_17_anos.csv'

try:
    with open(input_file_path, 'r', encoding='utf-8') as f:
        all_lines = f.readlines()
        
    # Linha 5 e 6 do arquivo (índice 4 e 5)
    header_l1_raw = all_lines[4] 
    header_l2_raw = all_lines[5] 

    # Corrigir as vírgulas no cabeçalho
    header_l1_fixed = header_l1_raw.replace('Até, cinco minutos', 'Até cinco minutos')
    header_l1_fixed = header_l1_fixed.replace('seis, minutos', 'seis minutos')
    header_l1_fixed = header_l1_fixed.replace('quinze, minutos', 'quinze minutos')
    header_l1_fixed = header_l1_fixed.replace('meia, hora', 'meia hora')
    header_l1_fixed = header_l1_fixed.replace('uma, hora', 'uma hora')
    header_l1_fixed = header_l1_fixed.replace('duas, horas', 'duas horas')
    header_l1_fixed = header_l1_fixed.replace('quatro, horas', 'quatro horas')

    
     #Processar as strings de cabeçalho
    header_l1 = pd.read_csv(io.StringIO(header_l1_fixed), header=None).iloc[0].ffill()
    header_l2 = pd.read_csv(io.StringIO(header_l2_raw), header=None).iloc[0]

    # Combinar os cabeçalhos
    idx_cols = ['Região Geográfica Intermediária', 'Grupo de idade', 'Cor ou raça']
    data_cols = [f"{l1} | {l2}" for l1, l2 in zip(header_l1[3:], header_l2[3:])]
    final_columns = idx_cols + data_cols

    # Carregar os dados
    df_desloc = pd.read_csv(
        input_file_path, 
        encoding='utf-8',
        skiprows=6, # Pula as 6 primeiras linhas
        header=None, 
        na_values=['-', '...'],
        engine='python'
    )
    
    df_desloc = df_desloc.iloc[:, :len(final_columns)]
    df_desloc.columns = final_columns

    
    # Aplicar ffill PRIMEIRO nas colunas de agrupamento
    # Isso preenche os valores 'Porto Velho' e '10 a 17 anos' nas linhas vazias abaixo deles
    cols_para_ffill = ['Região Geográfica Intermediária', 'Grupo de idade']
    df_desloc[cols_para_ffill] = df_desloc[cols_para_ffill].ffill()
    
    # AGORA, podemos remover com segurança as linhas onde 'Cor ou raça' é nulo
    # Isso limpa linhas de título/rodapé indesejadas sem perder dados
    df_desloc = df_desloc.dropna(subset=['Cor ou raça'])
    
    # Aplicar a limpeza de string (.str.strip())
    # Isso garante que ' 10 a 17 anos ' seja tratado como '10 a 17 anos'
    df_desloc['Grupo de idade'] = df_desloc['Grupo de idade'].astype(str).str.strip()

    # Conversão numérica (como antes)
    for col in data_cols:
        df_desloc[col] = pd.to_numeric(df_desloc[col], errors='coerce')

    # Esta verificação agora deve mostrar todos os grupos de idade
    print(df_desloc['Grupo de idade'].unique())

    
    idade_alvo = '10 a 17 anos'
    
    # O filtro agora é aplicado na coluna limpa e preenchida
    df_desloc_filtrado = df_desloc[df_desloc['Grupo de idade'] == idade_alvo].copy()
    
    if len(df_desloc_filtrado) == 0:
        print("AVISO: O filtro não retornou nenhuma linha. Verifique se '10 a 17 anos' está na lista acima.")
    else:
        print(f"Filtro aplicado com sucesso. {len(df_desloc_filtrado)} linhas encontradas.")
    
        # 8. Filtrar (Passo 7.5 do script original)
    idade_alvo = '10 a 17 anos'
    
    # O filtro agora é aplicado na coluna limpa
    df_desloc_filtrado = df_desloc[df_desloc['Grupo de idade'] == idade_alvo].copy()
    
    if len(df_desloc_filtrado) == 0:
        print("AVISO: O filtro não retornou nenhuma linha. O script pode falhar.")
    else:
        print(f"Filtro aplicado com sucesso. {len(df_desloc_filtrado)} linhas encontradas.")

    df_long = df_desloc_filtrado.melt(
        id_vars=idx_cols,
        var_name='Metrica_Combinada',
        value_name='Pessoas'
    )
    
    df_long[['Tempo de Deslocamento', 'Nível de Instrução']] = df_long['Metrica_Combinada'].str.split(' | ', expand=True, n=1)
    df_long = df_long.drop(columns=['Metrica_Combinada'])
    df_long = df_long.dropna(subset=['Pessoas'])

    print(f"Passo 10/10: Salvando arquivo como '{output_filename}'...")
    df_long.to_csv(output_filename, index=False)
    
    print("\n--- AVALIAÇÃO CONCLUÍDA COM SUCESSO ---")
    print(f"Arquivo '{output_filename}' salvo com sucesso. {len(df_long)} linhas.")
    
    print("\n--- Amostra do resultado final (Formato Longo) ---")
    print(df_long.head())
    
except Exception as e:
    print(f"\n--- ERRO DURANTE A AVALIAÇÃO ---")
    print(f"Ocorreu um erro: {e}")
    print("Por favor, revise o erro acima. O problema mais provável ainda é um 'ValueError' ou 'KeyError' se o filtro falhou.")

['Total' '10 a 17 anos' '18 a 64 anos' '65 anos ou mais']
Filtro aplicado com sucesso. 798 linhas encontradas.
Filtro aplicado com sucesso. 798 linhas encontradas.
Passo 10/10: Salvando arquivo como '../data/Processed/Deslocamento_processado_10_a_17_anos.csv'...

--- AVALIAÇÃO CONCLUÍDA COM SUCESSO ---
Arquivo '../data/Processed/Deslocamento_processado_10_a_17_anos.csv' salvo com sucesso. 14396 linhas.

--- Amostra do resultado final (Formato Longo) ---
  Região Geográfica Intermediária Grupo de idade Cor ou raça  Pessoas Tempo de Deslocamento Nível de Instrução
0                     Porto Velho   10 a 17 anos       Total   5254.0                 Total            | Total
1                     Porto Velho   10 a 17 anos      Branca   1134.0                 Total            | Total
2                     Porto Velho   10 a 17 anos       Preta    647.0                 Total            | Total
3                     Porto Velho   10 a 17 anos     Amarela     13.0                 Total       

## Confirmando a existência de dados somente de pessoas de 10 a 17 anos

In [13]:
import pandas as pd

# Nome do arquivo conforme enviado
file_path = '../data/Processed/Deslocamento_processado_10_a_17_anos.csv'

# A coluna 'Grupo de idade' é a segunda coluna, que tem índice 1
col_index_grupo_idade = 1

print(f"Iniciando a verificação dos valores únicos na coluna de índice {col_index_grupo_idade} do arquivo {file_path}...")

try:
    df_check = pd.read_csv(
        file_path,
        encoding='utf-8',  
        skiprows=6,        
        header=None,       
        na_values=['-', '...'], 
        engine='python'
    )
    
    unique_values = df_check[col_index_grupo_idade].dropna().unique()
    
    print("\n--- CONFIRMADO ---")
    print("Estes são os valores únicos exatos na coluna 'Grupo de idade':\n")
    for value in unique_values:
        print(f"'{value}'")

except FileNotFoundError:
    print(f"Erro: O arquivo {file_path} não foi encontrado.")
except Exception as e:
    print(f"Ocorreu um erro ao processar o arquivo: {e}")

Iniciando a verificação dos valores únicos na coluna de índice 1 do arquivo ../data/Processed/Deslocamento_processado_10_a_17_anos.csv...

--- CONFIRMADO ---
Estes são os valores únicos exatos na coluna 'Grupo de idade':

'10 a 17 anos'


### Corrigindo coluna Tempo de deslocamento

In [14]:
import pandas as pd
import numpy as np

print("--- Corrigindo 'Deslocamento_processado.csv' (Abordagem Correta) ---")

df = pd.read_csv('../data/Processed/Deslocamento_processado_10_a_17_anos.csv')
print("Dados originais (com erro):")
print(df.head())

df['Metrica_Combinada'] = (
    df['Tempo de Deslocamento'].astype(str) + 
    " " + 
    df['Nível de Instrução'].astype(str)
)

df = df.drop(columns=['Tempo de Deslocamento', 'Nível de Instrução'])

df[['Tempo de Deslocamento', 'Nível de Instrução']] = df['Metrica_Combinada'].str.split(r'\s*\|\s*', expand=True)
df = df.drop(columns=['Metrica_Combinada'])

df['Tempo de Deslocamento'] = df['Tempo de Deslocamento'].str.strip()
df['Nível de Instrução'] = df['Nível de Instrução'].str.strip()

print("\nDados corrigidos:")
df.to_csv('../data/Processed/Deslocamento_processado_corrigido.csv', index=False)
print(df.head())

--- Corrigindo 'Deslocamento_processado.csv' (Abordagem Correta) ---
Dados originais (com erro):
  Região Geográfica Intermediária Grupo de idade Cor ou raça  Pessoas Tempo de Deslocamento Nível de Instrução
0                     Porto Velho   10 a 17 anos       Total   5254.0                 Total            | Total
1                     Porto Velho   10 a 17 anos      Branca   1134.0                 Total            | Total
2                     Porto Velho   10 a 17 anos       Preta    647.0                 Total            | Total
3                     Porto Velho   10 a 17 anos     Amarela     13.0                 Total            | Total
4                     Porto Velho   10 a 17 anos       Parda   3430.0                 Total            | Total

Dados corrigidos:
  Região Geográfica Intermediária Grupo de idade Cor ou raça  Pessoas Tempo de Deslocamento Nível de Instrução
0                     Porto Velho   10 a 17 anos       Total   5254.0                 Total              To

In [ ]:
## Convertendo a tabela de enriquecimento para csv
# ! pip install xlrd

output_file = '../data/Raw/RELATORIO_DTB_BRASIL_2024_DISTRITOS.csv'
try:
    excel_file = '../data/Raw/RELATORIO_DTB_BRASIL_2024_DISTRITOS.xls'
    df = pd.read_excel(excel_file, sheet_name='DTB_Distritos', engine='xlrd')

    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"Arquivo '{output_file}' salvo com sucesso (codificação UTF-8)!")
except FileNotFoundError:
    print(f"Arquivo não encontrado: {excel_file}")
except Exception as e:
    print(f"Ocorreu um erro durante o processamento: {str(e)}")

# removendo o arquivo xls original
#! rm ../data/Raw/RELATORIO_DTB_BRASIL_2024_DISTRITOS.xls

## Enriquecendo a tabela de deslocamento

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

try:
    
    # Corrigindo o caminho do arquivo para o diretório atual
    file_mapa = '../data/Raw/RELATORIO_DTB_BRASIL_2024_DISTRITOS.csv'
    df_mapa = None

    try:
        print(f"--- Carregando mapa (DTB) '{file_mapa}' com UTF-8 ---")
        df_mapa = pd.read_csv(file_mapa, header=6, encoding='utf-8')
        print("Carregado com UTF-8 com sucesso.")
    
    except UnicodeDecodeError:
        print("Falha no UTF-8. Tentando 'latin-1' (padrão comum do IBGE)...")
        df_mapa = pd.read_csv(file_mapa, header=6, encoding='latin-1')
        print("Carregado com 'latin-1' com sucesso.")
    
    except Exception as e:
        print(f"Erro inesperado ao carregar o mapa (DTB): {e}")
        raise e


    df_mapa = df_mapa.loc[:, ~df_mapa.columns.str.contains('^Unnamed')]

    colunas_mapa = [
        'Região Geográfica Intermediária',      # <-- CÓDIGO da Região (ex: 1101)
        'Nome Região Geográfica Intermediária', # <-- NOME da Região (ex: Porto Velho)
        'UF',                                 # <-- CÓDIGO da UF (ex: 11)
        'Nome_UF'                             # <-- NOME da UF (ex: Rondônia)
    ]
    df_de_para = df_mapa[colunas_mapa].drop_duplicates().reset_index(drop=True)
    
    df_de_para = df_de_para.rename(columns={
        'Região Geográfica Intermediária': 'Código Região Intermediária',
        'Nome Região Geográfica Intermediária': 'Nome Região Intermediária (Chave)',
        'UF': 'Código UF',
        'Nome_UF': 'Nome UF'
    })

    print("\nMapa de Região para UF (com códigos) criado. Amostra:")
    print(df_de_para.head())

    file_data = '../data/Processed/Deslocamento_processado_corrigido.csv'
    print(f"\n--- Carregando dados (já corrigidos) '{file_data}' com UTF-8 ---")
    
    df_desloc_corrigido = pd.read_csv(file_data, encoding='utf-8')
    
    print("Arquivo de dados carregado. Amostra:")
    print(df_desloc_corrigido.head())

    print("\n--- Enriquecendo dados com a UF e Códigos (Merge) ---")
    
    df_final_enriquecido = pd.merge(
        df_desloc_corrigido,                      # Tabela da esquerda (dados)
        df_de_para,                               # Tabela da direita (mapa)
        left_on='Região Geográfica Intermediária',  # Chave na esquerda (Nome)
        right_on='Nome Região Intermediária (Chave)', # Chave na direita (Nome)
        how='left'                                # Manter todos os registros de Deslocamento
    )

    if 'Nome Região Intermediária (Chave)' in df_final_enriquecido.columns:
        df_final_enriquecido = df_final_enriquecido.drop(columns=['Nome Região Intermediária (Chave)'])

    df_final_enriquecido = df_final_enriquecido.rename(columns={'Nome UF': 'UF'})

    print("\n--- Dados Finais Enriquecidos com UF e Códigos ---")
    colunas_exibicao = ['Região Geográfica Intermediária', 'UF', 'Código UF', 'Código Região Intermediária', 'Pessoas']
    print(df_final_enriquecido[colunas_exibicao].head())
    
    null_uf_count = df_final_enriquecido['UF'].isnull().sum()
    if null_uf_count > 0:
        print(f"\n*** Atenção: {null_uf_count} linhas não encontraram uma UF correspondente. ***")
        print("Exemplos de nomes do arquivo 'Deslocamento' que falharam no merge:")
        print(df_final_enriquecido[df_final_enriquecido['UF'].isnull()]['Região Geográfica Intermediária'].unique()[:10])

    #transformar os códigos em inteiros
    df_final_enriquecido['Código UF'] = df_final_enriquecido['Código UF'].astype('Int64')
    df_final_enriquecido['Código Região Intermediária'] = df_final_enriquecido['Código Região Intermediária'].astype('Int64')

    output_file = '../data/Processed/Deslocamento_Enriquecido_com_UF_e_Codigos.csv'
    df_final_enriquecido.to_csv(output_file, index=False, encoding='utf-8')
    print(f"\nArquivo '{output_file}' salvo com sucesso (codificação UTF-8)!")

except FileNotFoundError as e:
    print(f"\n*** ERRO: Arquivo não encontrado ***")
    print(f"Não foi possível encontrar o arquivo: {e.filename}")
    print("Por favor, verifique se os nomes dos arquivos estão corretos e se eles foram carregados.")
except Exception as e:
    print(f"\nOcorreu um erro durante o processamento: {e}")

--- Carregando mapa (DTB) '../data/Raw/RELATORIO_DTB_BRASIL_2024_DISTRITOS.csv' com UTF-8 ---
Carregado com UTF-8 com sucesso.

Mapa de Região para UF (com códigos) criado. Amostra:
   Código Região Intermediária Nome Região Intermediária (Chave)  Código UF   Nome UF
0                         1102                         Ji-Paraná         11  Rondônia
1                         1101                       Porto Velho         11  Rondônia
2                         1201                        Rio Branco         12      Acre
3                         1202                   Cruzeiro do Sul         12      Acre
4                         1302                              Tefé         13  Amazonas

--- Carregando dados (já corrigidos) '../data/Processed/Deslocamento_processado_corrigido.csv' com UTF-8 ---
Arquivo de dados carregado. Amostra:
  Região Geográfica Intermediária Grupo de idade Cor ou raça  Pessoas Tempo de Deslocamento Nível de Instrução
0                     Porto Velho   10 a 17 

In [ ]:
### Criando colunas numéricas para 'Tempo de Deslocamento'
import pandas as pd
import numpy as np

try:
    df = pd.read_csv('../data/Processed/Deslocamento_Enriquecido_com_UF_e_Codigos.csv')
except FileNotFoundError:
    print("Erro: Arquivo não encontrado. Verifique o nome e o caminho do arquivo.")


if 'df' in locals():
    print("Arquivo carregado com sucesso.")


    
    mapeamento_tempo_medio = {
        'Total': np.nan,  # 'Total' é um agregado, não um tempo. np.nan é o ideal para ignorá-lo em cálculos.
        'Até cinco minutos': 2.5,                 # Ponto médio de (0 + 5) / 2
        'De seis minutos até quinze minutos': 10.5, # Ponto médio de (6 + 15) / 2
        'Mais de quinze minutos até meia hora': 23.0, # Ponto médio de (16 + 30) / 2
        'Mais de meia hora até uma hora': 45.5,    # Ponto médio de (31 + 60) / 2
        'Mais de uma hora até duas horas': 90.5,   # Ponto médio de (61 + 120) / 2
        'Mais de duas horas até quatro horas': 180.5, # Ponto médio de (121 + 240) / 2
        'Mais de quatro horas': 300.5                # ASSUMINDO um intervalo de 4 a 6 horas (241 a 360 min)
    }

    
    df['Tempo de Deslocamento Médio'] = df['Tempo de Deslocamento'].map(mapeamento_tempo_medio)

    print("\nVerificando a transformação (as 5 primeiras linhas com 'Total'):")
    print(df[['Tempo de Deslocamento', 'Tempo de Deslocamento Médio']].head())
    
    print("\nVerificando a transformação (alguns valores reais):")
    print(df[df['Tempo de Deslocamento'] != 'Total'][['Tempo de Deslocamento', 'Tempo de Deslocamento Médio']].head())

    output_filename = '../data/Processed/Deslocamento_Enriquecido_com_UF_e_Codigos.csv'
    df.to_csv(output_filename, index=False)
    
    print(f"\nArquivo modificado salvo com sucesso como: '{output_filename}'")

Arquivo carregado com sucesso.

Verificando a transformação (as 5 primeiras linhas com 'Total'):
  Tempo de Deslocamento  Tempo de Deslocamento Médio
0                 Total                          NaN
1                 Total                          NaN
2                 Total                          NaN
3                 Total                          NaN
4                 Total                          NaN

Verificando a transformação (alguns valores reais):
     Tempo de Deslocamento  Tempo de Deslocamento Médio
2578     Até cinco minutos                          2.5
2579     Até cinco minutos                          2.5
2580     Até cinco minutos                          2.5
2581     Até cinco minutos                          2.5
2582     Até cinco minutos                          2.5

Arquivo modificado salvo com sucesso como: '../data/Processed/Deslocamento_Enriquecido_com_UF_e_Codigos.csv'


### Remoção dos arquivos temporários

In [9]:
# removendo os arquivos temporários
! rm ../data/Processed/Deslocamento_processado_10_a_17_anos.csv
! rm ../data/Processed/Deslocamento_processado_corrigido.csv

In [ ]:
df = pd.read_csv('../data/Processed/Deslocamento_Enriquecido_com_UF_e_Codigos.csv')

# Filtra apenas as linhas onde 'Cor ou raça' é 'Total'
df_total = df[df['Cor ou raça'] == 'Total'].copy()

# Apaga a coluna 'Cor ou raça'
df_total = df_total.drop(columns=['Cor ou raça'])

# Transofrma a coluna 'Codigo UF' em tipo inteiro
df_total['Código UF'] = df_total['Código UF'].astype('Int64')
df_total['Código Região Intermediária'] = df_total['Código Região Intermediária'].astype('Int64')

# Salva o DataFrame filtrado em um novo arquivo CSV
df_total.to_csv('../data/Processed/Deslocamento_Enriquecido_com_UF_e_Codigos.csv', index=False)


print("Arquivo atualizado para conter apenas o total de pessoas por região.")

Arquivo atualizado para conter apenas o total de pessoas por região.


### Limpeza dos dados Saneamento.csv e indicesEnsino.csv

Essas tabelas não demandam pré-processamento pois são advindas do BigQuerry do Google e também já foram pré-processadas pela "Base dos Dados"

In [ ]:
SELECT
    dados.ano AS ano,
    dados.sigla_uf AS sigla_uf,
    diretorio_sigla_uf.nome AS sigla_uf_nome,
    -- Agregando os dados (SOMA para populações, volumes e investimentos)
    SUM(dados.populacao_atendida_agua) AS populacao_atendida_agua,
    SUM(dados.populacao_atentida_esgoto) AS populacao_atentida_esgoto,
    SUM(dados.volume_agua_produzido) AS volume_agua_produzido,
    SUM(dados.volume_agua_tratada_eta) AS volume_agua_tratada_eta,
    -- Usando MÉDIA para os índices (faz mais sentido que somar)
    AVG(dados.indice_coleta_esgoto) AS indice_coleta_esgoto,
    AVG(dados.indice_tratamento_esgoto) AS indice_tratamento_esgoto,
    -- Agregando os investimentos
    SUM(dados.investimento_esgoto_estado) AS investimento_esgoto_estado,
    SUM(dados.investimento_recurso_proprio_estado) AS investimento_recurso_proprio_estado,
    SUM(dados.investimento_total_estado) AS investimento_total_estado
    
FROM
    `basedosdados.br_mdr_snis.municipio_agua_esgoto` AS dados
LEFT JOIN
    (SELECT DISTINCT sigla, nome FROM `basedosdados.br_bd_diretorios_brasil.uf`) AS diretorio_sigla_uf
ON
    dados.sigla_uf = diretorio_sigla_uf.sigla
WHERE
    dados.ano = 2022
    
GROUP BY
    dados.ano,
    dados.sigla_uf,
    diretorio_sigla_uf.nome

In [ ]:
SELECT
    dados.ano as ano,
    dados.sigla_uf as sigla_uf,
    dados.localizacao as localizacao,
    dados.rede as rede,
    dados.taxa_promocao_ef as taxa_promocao_ef,
    dados.taxa_promocao_ef_5_ano as taxa_promocao_ef_5_ano,
    dados.taxa_promocao_ef_6_ano as taxa_promocao_ef_6_ano,
    dados.taxa_promocao_ef_7_ano as taxa_promocao_ef_7_ano,
    dados.taxa_promocao_ef_8_ano as taxa_promocao_ef_8_ano,
    dados.taxa_promocao_ef_9_ano as taxa_promocao_ef_9_ano,
    dados.taxa_promocao_em as taxa_promocao_em,
    dados.taxa_promocao_em_1_ano as taxa_promocao_em_1_ano,
    dados.taxa_promocao_em_2_ano as taxa_promocao_em_2_ano,
    dados.taxa_promocao_em_3_ano as taxa_promocao_em_3_ano,
    dados.taxa_repetencia_ef as taxa_repetencia_ef,
    dados.taxa_repetencia_ef_5_ano as taxa_repetencia_ef_5_ano,
    dados.taxa_repetencia_ef_6_ano as taxa_repetencia_ef_6_ano,
    dados.taxa_repetencia_ef_7_ano as taxa_repetencia_ef_7_ano,
    dados.taxa_repetencia_ef_8_ano as taxa_repetencia_ef_8_ano,
    dados.taxa_repetencia_ef_9_ano as taxa_repetencia_ef_9_ano,
    dados.taxa_repetencia_em as taxa_repetencia_em,
    dados.taxa_repetencia_em_1_ano as taxa_repetencia_em_1_ano,
    dados.taxa_repetencia_em_2_ano as taxa_repetencia_em_2_ano,
    dados.taxa_repetencia_em_3_ano as taxa_repetencia_em_3_ano,
    dados.taxa_evasao_ef as taxa_evasao_ef,
    dados.taxa_evasao_ef_5_ano as taxa_evasao_ef_5_ano,
    dados.taxa_evasao_ef_6_ano as taxa_evasao_ef_6_ano,
    dados.taxa_evasao_ef_7_ano as taxa_evasao_ef_7_ano,
    dados.taxa_evasao_ef_8_ano as taxa_evasao_ef_8_ano,
    dados.taxa_evasao_ef_9_ano as taxa_evasao_ef_9_ano,
    dados.taxa_evasao_em as taxa_evasao_em,
    dados.taxa_evasao_em_1_ano as taxa_evasao_em_1_ano,
    dados.taxa_evasao_em_2_ano as taxa_evasao_em_2_ano,
    dados.taxa_evasao_em_3_ano as taxa_evasao_em_3_ano
FROM `basedosdados.br_inep_indicadores_educacionais.uf_taxa_transicao` AS dados
WHERE dados.ano = 2022;

In [ ]:
# Move os arquivos Saneamento.csv e indicesEnsino.csv para a pasta de dados limpos
# ! mv ../data/Raw/Saneamento.csv ../data/Processed/Saneamento.csv
# ! mv ../data/Raw/indicesEnsino.csv ../data/Processed/indicesEnsino.csv